#Ingerir o arquivo results.json
1- Ler o arquivo usando a API de leitura de DataFrame do Spark

2-  Definir e aplicar o Schema 

3- Adicionar colunas de metadados
- Arquivo de origem
- Timestamp (data/hora) de ingestão

4- Escrever/salvar na tabela Delta da camada bronze

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
source_file = f"{landing_folder_path}/{v_batch_id}/results"
table_name = f"{catalog_name}.{bronze_schema}.results"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DateType, IntegerType, DoubleType

results_schema = StructType ([
    StructField ('constructorId', StringType()),
    StructField ('date', DateType()),
    StructField ('driverId', StringType()),
    StructField ('grid', IntegerType()),
    StructField ('laps', IntegerType()),
    StructField ('number', IntegerType()),
    StructField ('points', DoubleType()),
    StructField ('position', IntegerType()),
    StructField ('positionText', StringType()),
    StructField ('raceName', StringType()),
    StructField ('round', IntegerType()),
    StructField ('season', IntegerType()),
    StructField ('status', StringType()),
    StructField ('url', StringType()) 
])

In [0]:
results_df = (
    spark.read
    .format('json')
    .option('mode', 'FAILFAST')
    .schema(results_schema)
    .load(source_file)
)

In [0]:
results_final_df = add_ingestion_metadata(results_df)

In [0]:
display(results_final_df)

In [0]:
write_to_bronze(
    input_df = results_final_df,
    target_name = table_name,
    batch_id = v_batch_id
)

In [0]:
display(spark.read.table(table_name))

constructorId,date,driverId,grid,laps,number,points,position,positionText,raceName,round,season,status,url,ingestion_timestamp,source_file,batch_id
red_bull,2024-03-02,max_verstappen,1,57,1,26.0,1,1,bahrain grand prix,1,2024,Finished,https://en.wikipedia.org/wiki/2024_Bahrain_Grand_Prix,2026-09-11T22:57:12.449Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/results/results_2024.json,2025-01
red_bull,2024-03-02,perez,5,57,11,18.0,2,2,bahrain grand prix,1,2024,Finished,https://en.wikipedia.org/wiki/2024_Bahrain_Grand_Prix,2026-09-11T22:57:12.449Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/results/results_2024.json,2025-01
ferrari,2024-03-02,sainz,4,57,55,15.0,3,3,bahrain grand prix,1,2024,Finished,https://en.wikipedia.org/wiki/2024_Bahrain_Grand_Prix,2026-09-11T22:57:12.449Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/results/results_2024.json,2025-01
ferrari,2024-03-02,leclerc,2,57,16,12.0,4,4,bahrain grand prix,1,2024,Finished,https://en.wikipedia.org/wiki/2024_Bahrain_Grand_Prix,2026-09-11T22:57:12.449Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/results/results_2024.json,2025-01
mercedes,2024-03-02,russell,3,57,63,10.0,5,5,bahrain grand prix,1,2024,Finished,https://en.wikipedia.org/wiki/2024_Bahrain_Grand_Prix,2026-09-11T22:57:12.449Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/results/results_2024.json,2025-01
mclaren,2024-03-02,norris,7,57,4,8.0,6,6,bahrain grand prix,1,2024,Finished,https://en.wikipedia.org/wiki/2024_Bahrain_Grand_Prix,2026-09-11T22:57:12.449Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/results/results_2024.json,2025-01
mercedes,2024-03-02,hamilton,9,57,44,6.0,7,7,bahrain grand prix,1,2024,Finished,https://en.wikipedia.org/wiki/2024_Bahrain_Grand_Prix,2026-09-11T22:57:12.449Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/results/results_2024.json,2025-01
mclaren,2024-03-02,piastri,8,57,81,4.0,8,8,bahrain grand prix,1,2024,Finished,https://en.wikipedia.org/wiki/2024_Bahrain_Grand_Prix,2026-09-11T22:57:12.449Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/results/results_2024.json,2025-01
aston_martin,2024-03-02,alonso,6,57,14,2.0,9,9,bahrain grand prix,1,2024,Finished,https://en.wikipedia.org/wiki/2024_Bahrain_Grand_Prix,2026-09-11T22:57:12.449Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/results/results_2024.json,2025-01
aston_martin,2024-03-02,stroll,12,57,18,1.0,10,10,bahrain grand prix,1,2024,Finished,https://en.wikipedia.org/wiki/2024_Bahrain_Grand_Prix,2026-09-11T22:57:12.449Z,dbfs:/Volumes/formula1_incr/landing/arquivos/2025-01/results/results_2024.json,2025-01


In [0]:
%sql
SELECT season, COUNT(*)
FROM formula1.bronze.results
GROUP BY season
ORDER BY season; 